Some imports and paths

In [128]:
import os
import pandas as pd
from icecream import ic
from pathlib import Path
from pandas import DataFrame

THIS_FILE_PATH = Path(os.getcwd()) / "olistbr.ipynb"

THIS_PROJECT_PATH = THIS_FILE_PATH.parent.parent
DATA_RAW_PATH = THIS_PROJECT_PATH / "data" / "raw"

if not DATA_RAW_PATH.exists():
    raise FileExistsError()

ic(THIS_PROJECT_PATH)
ic(THIS_FILE_PATH)
ic(DATA_RAW_PATH)

ic| THIS_PROJECT_PATH: WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr')
ic| THIS_FILE_PATH: WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr/notebooks/olistbr.ipynb')
ic| DATA_RAW_PATH: WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr/data/raw')


WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr/data/raw')

In [129]:
raw_filenames: list[str] = os.listdir(DATA_RAW_PATH)
raw_filepaths: list[Path] = [DATA_RAW_PATH / f for f in raw_filenames]
# ic(raw_filenames)
# ic(raw_filepaths)

dfs: dict[str, DataFrame] = {f.name: pd.read_csv(f, dtype=str) for f in raw_filepaths}
# ic(dfs)

Measure Functions

In [130]:
def lev_d(a: str, b: str) -> int:
    """Levenshtein distance
    time: O(mn) quadratic
    """
    lena = len(a)
    lenb = len(b)

    if lenb == 0:
        return lena

    if lena == 0:
        return lenb

    heada = a[0]
    headb = b[0]
    taila = a[1:]
    tailb = b[1:]

    if heada == headb:
        return lev_d(taila, tailb)

    return 1 + min(lev_d(taila, b), lev_d(a, tailb), lev_d(taila, tailb))


def sim_norm_lev_d(a: str, b: str) -> float:
    """similarity normalised lev_d"""
    lena = len(a)
    lenb = len(b)

    return 1 - lev_d(a, b) / max(lena, lenb)


def dice_sim(A: set[str], B: set[str]) -> float:
    """Dice similarity
    |AnB| / (|A|+|B|/2) = 2*|AnB| / (|A|+|B|)
    """
    return 2 * len(A.intersection(B)) / (len(A) + len(B))

Inference Function

In [ ]:
import string


def infer_dtypes(df: DataFrame) -> DataFrame:
    """
    df -> out_df
    infers a column datatype from its attributes
    """

    def is_chars_in_string(chars: str, parent_string: str) -> bool:
        if set(chars).intersection(set(parent_string)):
            return True

        return False

    # TODO: add chars_used_subset_of_hex_digits
    out_df = pd.DataFrame(
        index=df.columns,
        columns=[
            # keys
            "has_unique_entries",
            # nulls
            "has_nulls",
            "where_nulls",
            "total_nulls",
            # chars
            "sorted_chars_used",
            "total_unique_chars_used",
            "has_ascii",
            "has_non_ascii",
            "has_prefix_zero",
            "dice_sim_to_ascii",
            "dice_sim_to_non_ascii",
            "min_str_value",
            "max_str_value",
            "entry_lengths",
            "total_unique_entry_lengths",
            "max_entry_length",
            "is_fixed_length",
            # numeric
            "chars_used_subset_of_numeric",
            "has_prefix_dash",
            "has_digits",
            "has_hex_digits",
            "has_decimal",
            "dice_sim_to_digits",
            "dice_sim_to_hex_digits",
            "min_numeric_value",
            "max_numeric_value",
            # datetime
            "has_dash",
            "has_colon",
            "has_space",
            # bit
            "has_exactly_two_entries",
        ],
    )

    cols = df.columns

    for col in cols:
        print(col)

        # global vars
        clean_series = df[col].dropna()
        chars_used: set[str] = set("".join(clean_series.astype(str)))
        sorted_chars_used: str = "".join(sorted(chars_used))

        # keys ==================================================
        out_df.loc[col, "has_unique_entries"] = 1 if clean_series.is_unique else 0

        # nulls ==================================================
        where_null = df[col].isnull()

        out_df.loc[col, "has_nulls"] = 1 if where_null.any() else 0
        out_df.loc[col, "where_nulls"] = df[where_null].index.tolist()
        out_df.loc[col, "total_nulls"] = where_null.sum()

        # chars ==================================================
        # if col chars > ascii when col chars - ascii > 0
        excess_ascii: set[str] = chars_used - set(string.printable)
        entry_lengths = sorted(clean_series.astype(str).str.len().unique().tolist())

        out_df.loc[col, "sorted_chars_used"] = sorted_chars_used
        out_df.loc[col, "total_unique_chars_used"] = len(sorted_chars_used)
        out_df.loc[col, "has_ascii"] = (
            1
            if is_chars_in_string(
                string.printable,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_non_ascii"] = 1 if excess_ascii else 0
        out_df.loc[col, "has_prefix_zero"] = (
            1 if clean_series.astype(str).str.startswith("0").any() else 0
        )
        out_df.loc[col, "dice_sim_to_ascii"] = dice_sim(
            set(string.printable),
            chars_used,
        )
        out_df.loc[col, "dice_sim_to_non_ascii"] = dice_sim(
            excess_ascii,
            chars_used,
        )
        out_df.loc[col, "min_str_value"] = min(clean_series)
        out_df.loc[col, "max_str_value"] = max(clean_series)
        out_df.loc[col, "entry_lengths"] = entry_lengths
        out_df.loc[col, "total_unique_entry_lengths"] = len(entry_lengths)
        out_df.loc[col, "max_entry_length"] = max(entry_lengths)
        out_df.loc[col, "is_fixed_length"] = 1 if len(entry_lengths) == 1 else 0

        # numeric ==================================================
        str_numeric = string.digits + "-."

        out_df.loc[col, "chars_used_subset_of_numeric"] = (
            1 if chars_used.issubset(str_numeric) else 0
        )
        out_df.loc[col, "has_prefix_dash"] = (
            1 if clean_series.astype(str).str.startswith("-").any() else 0
        )
        out_df.loc[col, "has_digits"] = (
            1
            if is_chars_in_string(
                string.digits,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_hex_digits"] = (
            1
            if is_chars_in_string(
                string.hexdigits,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_decimal"] = (
            1
            if is_chars_in_string(
                ".",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "dice_sim_to_digits"] = dice_sim(
            set(string.digits),
            chars_used,
        )
        out_df.loc[col, "dice_sim_to_hex_digits"] = dice_sim(
            set(string.hexdigits),
            chars_used,
        )

        val = pd.to_numeric(clean_series, errors="coerce")
        numeric_min = val.min()
        numeric_max = val.max()
        out_df.loc[col, "min_numeric_value"] = (
            numeric_min if pd.notnull(numeric_min) else pd.NA
        )
        out_df.loc[col, "max_numeric_value"] = (
            numeric_max if pd.notnull(numeric_max) else pd.NA
        )

        # datetime ==================================================
        out_df.loc[col, "has_dash"] = (
            1
            if is_chars_in_string(
                "-",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_colon"] = (
            1
            if is_chars_in_string(
                ":",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_space"] = (
            1
            if is_chars_in_string(
                " ",
                sorted_chars_used,
            )
            else 0
        )

        # bit ==================================================
        unique_entries = clean_series.unique()
        out_df.loc[col, "has_exactly_two_entries"] = (
            1 if len(unique_entries) == 2 else 0
        )
    return out_df

In [145]:
pd.set_option("display.max_columns", None)

filename = raw_filenames[6]
filename = "olist_" + "closed_deals" + "_dataset.csv"
# filename = 'product_category_name_translation.csv'
print(filename)

df = dfs[filename]
df_attributes = infer_dtypes(df)
df_attributes

olist_closed_deals_dataset.csv
mql_id
seller_id
sdr_id
sr_id
won_date
business_segment
lead_type
lead_behaviour_profile
has_company
has_gtin
average_stock
business_type
declared_product_catalog_size
declared_monthly_revenue


,has_unique_entries,has_nulls,where_nulls,total_nulls,sorted_chars_used,total_unique_chars_used,has_ascii,has_non_ascii,has_prefix_zero,dice_sim_to_ascii,dice_sim_to_non_ascii,min_str_value,max_str_value,entry_lengths,total_unique_entry_lengths,max_entry_length,is_fixed_length,chars_used_subset_of_numeric,has_prefix_dash,has_digits,has_hex_digits,has_decimal,dice_sim_to_digits,dice_sim_to_hex_digits,min_numeric_value,max_numeric_value,has_dash,has_colon,has_space,has_exactly_two_entries
mql_id,1,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,000dd3543ac84d906eae52e7c779bb2a,fff8db9478d2fd72df65a67ee6b62f67,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
seller_id,1,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,00065220becb8785e2cf78355eb9bf68,ffc470761de7d0232558ba5e786e57b7,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
sdr_id,0,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,068066e24f0c643eb1d089c7dd20cd73,fdb16d3cbbeb5798f2f66c4096be026d,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
sr_id,0,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,060c0a26f19f4d66b42e0d8796688490,fbf4aef3f6915dc0c3c97d6812522f6a,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
won_date,0,0,[],0,-0123456789:,13,1,0,0,0.230088,0.0,2017-12-05 02:00:00,2018-11-14 18:04:19,[19],1,19,1,0,0,1,1,0,0.869565,0.571429,<NA>,<NA>,1,1,1,0
business_segment,0,1,[186],1,_abcdefghijklmnoprstuvwy,24,1,0,0,0.387097,0.0,air_conditioning,watches,"[3, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17...",17,31,0,0,0,0,1,0,0.0,0.26087,<NA>,<NA>,0,0,0,0
lead_type,0,1,"[96, 393, 445, 447, 594, 829]",6,_abdefghilmnoprstuy,19,1,0,0,0.319328,0.0,industry,other,"[5, 7, 8, 10, 12, 13, 15]",7,15,0,0,0,0,1,0,0.0,0.243902,<NA>,<NA>,0,0,0,0
lead_behaviour_profile,0,1,"[3, 5, 17, 20, 22, 23, 24, 25, 26, 38, 39, 47,...",177,",acefghklorstw",15,1,0,0,0.26087,0.0,cat,wolf,"[3, 4, 5, 9, 10, 11]",6,11,0,0,0,0,1,0,0.0,0.216216,<NA>,<NA>,0,0,1,0
has_company,0,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",779,FTaelrsu,8,1,0,0,0.148148,0.0,False,True,"[4, 5]",2,5,0,0,0,0,1,0,0.0,0.2,<NA>,<NA>,0,0,0,1
has_gtin,0,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",778,FTaelrsu,8,1,0,0,0.148148,0.0,False,True,"[4, 5]",2,5,0,0,0,0,1,0,0.0,0.2,<NA>,<NA>,0,0,0,1


In [133]:
"""scratch
inclusive
tinyint:     2**0-1 to 2**8-1
smallint:   -2**15  to 2**15-1
int:        -2**31  to 2**31-1
bigint:     -2**63  to 2**63-1

"""


"""
different known datatypes

text
numeric
datetime
bit

assume is text
(n, var, char) (#)
    'n' if has_non_ascii else ''
    '' if is_fixed_length else 'var'
    'char'
    # = max_entry_length

assume is int
(tiny, small, int, big)
    has_prefix_dash
    M = max(abs(min_numeric_value), abs(max_numeric_value))
    
    if min_numeric_value is negative: cannot be tinyint

    if M <= 2**8-1:


    


    

decimal(p,s)
datetime2
bit

not null
primary key

"""
print(
    df_attributes[
        [
            "chars_used_subset_of_numeric",
            "has_prefix_dash",
            "has_digits",
            "has_hex_digits",
            "has_decimal",
            "dice_sim_to_digits",
            "dice_sim_to_hex_digits",
            "min_numeric_value",
            "max_numeric_value",
        ]
    ]
)
df_attributes

                         chars_used_subset_of_numeric has_prefix_dash  \
customer_id                                         0               0   
customer_unique_id                                  0               0   
customer_zip_code_prefix                            1               0   
customer_city                                       0               0   
customer_state                                      0               0   

                         has_digits has_hex_digits has_decimal  \
customer_id                       1              1           0   
customer_unique_id                1              1           0   
customer_zip_code_prefix          1              1           0   
customer_city                     1              1           0   
customer_state                    0              1           0   

                         dice_sim_to_digits dice_sim_to_hex_digits  \
customer_id                        0.769231               0.842105   
customer_unique_id      

,has_unique_entries,has_nulls,where_nulls,total_nulls,sorted_chars_used,total_unique_chars_used,has_ascii,has_non_ascii,has_prefix_zero,dice_sim_to_ascii,dice_sim_to_non_ascii,min_str_value,max_str_value,entry_lengths,total_unique_entry_lengths,max_entry_length,is_fixed_length,chars_used_subset_of_numeric,has_prefix_dash,has_digits,has_hex_digits,has_decimal,dice_sim_to_digits,dice_sim_to_hex_digits,min_numeric_value,max_numeric_value,has_dash,has_colon,has_space,has_exactly_two_entries
customer_id,1,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
customer_unique_id,0,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,0000366f3b9a7992bf8c76cfdf3221e2,ffffd2657e2aad2907e67c3e9daecbeb,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
customer_zip_code_prefix,0,0,[],0,0123456789,10,1,0,1,0.181818,0.0,01003,99990,[5],1,5,1,1,0,1,1,0,1.0,0.625,1003,99990,0,0,0,0
customer_city,0,0,[],0,'-14abcdefghijklmnopqrstuvwxyz,31,1,0,0,0.473282,0.0,abadia dos dourados,zortea,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",28,32,0,0,0,1,1,0,0.097561,0.301887,<NA>,<NA>,1,0,1,0
customer_state,0,0,[],0,ABCDEFGIJLMNOPRST,17,1,0,0,0.290598,0.0,AC,TO,[2],1,2,1,0,0,0,1,0,0.0,0.307692,<NA>,<NA>,0,0,0,0


In [134]:
a = -101.466766
b = 121.105394

max(abs(a), abs(b))

121.105394

In [135]:
months = {
    "january",
    "february",
    "march",
    "april",
    "may",
    "june",
    "july",
    "august",
    "september",
    "october",
    "november",
    "december",
}

print(sorted(set(string.ascii_lowercase).difference(set("".join(months)))))
print(set(months))
print(set("".join(months)))

['k', 'q', 'w', 'x', 'z']
{'april', 'september', 'october', 'december', 'july', 'march', 'august', 'november', 'june', 'may', 'january', 'february'}
{'d', 'b', 'm', 'o', 'f', 't', 'u', 'v', 's', 'g', 'a', 'j', 'n', 'h', 'r', 'y', 'e', 'p', 'i', 'c', 'l'}


Old code (KEEP)

In [136]:
# dfs.keys()


# def describe_table(current_table: str, current_column: str) -> None:
#     df = dfs[current_table]

#     print(f"current column: {current_column}")
#     print("")
#     print(df[current_column].sample(5))
#     print("")
#     print(f"dtype:  {df[current_column].dtype}")
#     print(f"table:  {current_table}")
#     print(f"column: {current_column}")
#     print("")

#     # check if contains null
#     has_null = df[current_column].isnull().any()
#     print(f"contains NULL:  {has_null}")

#     # if has_null:
#     #     # filter out null
#     #     print("contains null")
#     #     df_notnull = df[df[current_column].notnull()]
#     #     pass

#     if df[current_column].dtype != "object":
#         print("not an object")

#         df[current_column] = df[current_column].dropna()

#         print(df[current_column].describe())

#         return

#     # check if requires 'n' prefix: nchar, nvarchar
#     # n = national, means contains unicode
#     is_national: bool = not df[current_column].apply(lambda x: str(x).isascii()).all()

#     # check if char or varchar
#     entry_lengths = df[current_column].dropna().str.len().unique()

#     max_length = entry_lengths.max()
#     is_fixed_length = entry_lengths.size == 1

#     # print(f'is char:        {is_char}')
#     print(f"is above ascii: {is_national}")
#     print(f"entry lengths:  {entry_lengths}")
#     print(f"max length:     {max_length}")
#     print(f"fixed length:   {is_fixed_length}")

# table = "olist_geolocation_dataset.csv"
# column = "geolocation_state"
# describe_table(current_table=table, current_column=column)